In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repo (or pull latest changes)
import os
repo_path = '/content/darts-cards'
if not os.path.exists(repo_path):
    !git clone https://github.com/javiergarciaduran/darts-cards.git {repo_path}
else:
    !git -C {repo_path} pull
%cd {repo_path}

In [ ]:
# 3. Install dependencies
!pip install -q optuna graphviz wandb
!pip install -q tensorboardX
!apt-get install -q graphviz

In [ ]:
# 4. Fix encoding (datasets/__init__.py may be UTF-16 on Windows)
!iconv -f UTF-16 -t UTF-8 /content/darts-cards/datasets/__init__.py > temp.py && mv temp.py /content/darts-cards/datasets/__init__.py

In [ ]:
# 5. Symlink dataset from Drive and verify
!mkdir -p ./data
!ln -sfn /content/drive/MyDrive/cards ./data/cards

from datasets.cards import get_cards
tr, nc = get_cards('./data/cards', split='train')
va, _  = get_cards('./data/cards', split='val')
print(f'train={len(tr)}  val={len(va)}  classes={nc}')

In [ ]:
# 6. Create output directories on Drive
!mkdir -p /content/drive/MyDrive/darts_experiments/hpo_random_search
!mkdir -p /content/drive/MyDrive/darts_experiments/hpo_tpe
!mkdir -p /content/drive/MyDrive/darts_experiments/hpo_logs
print('Output directories ready.')

## Random Search (baseline)

Runs `--n_trials` augment.py trials with hyperparameters sampled uniformly at random.
Each trial trains for **50 epochs** (fast proxy for final accuracy).

Results are appended to the CSV on Drive after every trial — safe to interrupt and resume.

In [ ]:
!python hpo_random_search_baseline.py \
    --n_trials 20 \
    --hpo_seed 42 \
    --hpo_output_dir /content/drive/MyDrive/darts_experiments/hpo_random_search \
    --csv_path /content/drive/MyDrive/darts_experiments/hpo_results_rs.csv \
    2>&1 | tee /content/drive/MyDrive/darts_experiments/hpo_logs/random_search.log

## TPE Search

Runs `--n_trials` augment.py trials guided by a multivariate TPE sampler.
The first trial always uses the known-good baseline config from `cards_augment_v1`
(lr=0.025, wd=3e-4, drop_path=0.2, aux=0.4, cutout=8) to warm-start the surrogate model.

Results are appended to the CSV on Drive after every trial — safe to interrupt and resume.

In [ ]:
!python hpo_tpe.py \
    --n_trials 20 \
    --hpo_seed 42 \
    --hpo_output_dir /content/drive/MyDrive/darts_experiments/hpo_tpe \
    --csv_path /content/drive/MyDrive/darts_experiments/hpo_tpe_results.csv \
    2>&1 | tee /content/drive/MyDrive/darts_experiments/hpo_logs/tpe.log

## Results Analysis

Load both CSVs from Drive, print the best trial for each strategy, and plot the optimisation history.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

RS_CSV  = '/content/drive/MyDrive/darts_experiments/hpo_results_rs.csv'
TPE_CSV = '/content/drive/MyDrive/darts_experiments/hpo_tpe_results.csv'

rs  = pd.read_csv(RS_CSV)
tpe = pd.read_csv(TPE_CSV)

# Filter out failed trials (value == -1)
rs_ok  = rs[rs['value'] != -1.0].copy()
tpe_ok = tpe[tpe['value'] != -1.0].copy()

print('=== Random Search ===')
print(f'Completed trials : {len(rs_ok)} / {len(rs)}')
best_rs = rs_ok.loc[rs_ok['value'].idxmax()]
print(f'Best Prec@1      : {best_rs["value"]*100:.2f}%')
print(best_rs.to_string())

print('\n=== TPE ===')
print(f'Completed trials : {len(tpe_ok)} / {len(tpe)}')
best_tpe = tpe_ok.loc[tpe_ok['value'].idxmax()]
print(f'Best Prec@1      : {best_tpe["value"]*100:.2f}%')
print(best_tpe.to_string())

# Running best (cumulative max) for each strategy
rs_ok['best_so_far']  = rs_ok['value'].cummax()
tpe_ok['best_so_far'] = tpe_ok['value'].cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: optimisation history (individual trials + running best)
ax = axes[0]
ax.scatter(rs_ok['trial_number'],  rs_ok['value'],  alpha=0.6, label='RS trials',  color='steelblue')
ax.scatter(tpe_ok['trial_number'], tpe_ok['value'], alpha=0.6, label='TPE trials', color='darkorange')
ax.plot(rs_ok['trial_number'],  rs_ok['best_so_far'],  '--', color='steelblue',  label='RS best so far')
ax.plot(tpe_ok['trial_number'], tpe_ok['best_so_far'], '--', color='darkorange', label='TPE best so far')
ax.set_xlabel('Trial number')
ax.set_ylabel('Validation Prec@1')
ax.set_title('Optimisation history')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: best hyperparameter values side-by-side
ax = axes[1]
params = ['lr', 'weight_decay', 'drop_path_prob', 'aux_weight', 'cutout_length']
x = range(len(params))
bar_w = 0.35
ax.bar([i - bar_w/2 for i in x], [best_rs[p]  for p in params], bar_w, label='RS best',  color='steelblue',  alpha=0.8)
ax.bar([i + bar_w/2 for i in x], [best_tpe[p] for p in params], bar_w, label='TPE best', color='darkorange', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(params, rotation=25, ha='right')
ax.set_title('Best hyperparameters')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plot_path = '/content/drive/MyDrive/darts_experiments/hpo_logs/hpo_comparison.png'
plt.savefig(plot_path, dpi=150)
print(f'\nSaved plot → {plot_path}')
plt.show()